In [ ]:
# Artem Pereyaslavtsev. homeguard_system

In [4]:
"""
HomeGuard Security System Simulator
Author: [Your Name]
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""

import random
from datetime import datetime
import time

# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
current_mode = "AWAY"

# Helper functions
def generate_reading_by_type(sensor_type):
    if sensor_type == "temperature":
        return random.randint(30, 100)
    elif sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
    return None

def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["type"]
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95:
            return True
    elif sensor_type == "motion":
        if reading_value:
            return True
    elif sensor_type == "door":
        if reading_value == "OPEN":
            return True
    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            return True
    return False

def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    # Security alerts
    if system_mode == "AWAY":
        if sensor["type"] == "motion" and reading_value:
            alerts.append({"severity": "HIGH", "message": f"SECURITY: Motion detected in {sensor['location']} while in AWAY mode!", "sensor_id": sensor["id"], "timestamp": timestamp})
        if sensor["type"] == "door" and reading_value == "OPEN":
            alerts.append({"severity": "HIGH", "message": f"SECURITY: {sensor['location']} opened while in AWAY mode!", "sensor_id": sensor["id"], "timestamp": timestamp})
    # Safety alerts
    if sensor["type"] == "temperature" and is_abnormal_reading(sensor, reading_value):
        alerts.append({"severity": "HIGH", "message": f"SAFETY: Temperature in {sensor['location']} abnormal: {reading_value}°F", "sensor_id": sensor["id"], "timestamp": timestamp})
    if sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append({"severity": "CRITICAL", "message": f"FIRE ALERT: Smoke detected in {sensor['location']}!", "sensor_id": sensor["id"], "timestamp": timestamp})
    # Comfort notifications
    if system_mode == "HOME" and sensor["type"] == "temperature" and not (65 <= reading_value <= 75):
        alerts.append({"severity": "LOW", "message": f"COMFORT: Temperature in {sensor['location']} out of comfort range: {reading_value}°F", "sensor_id": sensor["id"], "timestamp": timestamp})
    return alerts

def trigger_alert(alert):
    severity_symbol = {"LOW": "ℹ️", "MEDIUM": "⚠️", "HIGH": "🚨", "CRITICAL": "🔥"}
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

# Sensor class
class Sensor:
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None

    def read(self):
        self.current_value = generate_reading_by_type(self.type)
        return self.current_value

    def isAbnormal(self):
        sensor_dict = {"id": self.id, "location": self.location, "type": self.type, "threshold": self.threshold}
        return is_abnormal_reading(sensor_dict, self.current_value)

    def reset(self):
        self.current_value = None

    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

# Main simulation loop
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]

    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        for sensor in sensors:
            reading = sensor.read()
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

            sensor_dict = {"id": sensor.id, "location": sensor.location, "type": sensor.type, "threshold": sensor.threshold}
            alerts = process_reading(sensor_dict, reading, system_mode)
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

        time.sleep(1)  # delay to make output readable

# Run the simulation with at least 3 iterations
if __name__ == "__main__":
    run_simulation(duration_minutes=3, system_mode="AWAY")


=== HomeGuard Security System ===
Mode: AWAY


Time: 15:15:38
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [15:15:38] Sending notification to homeowner...
[READING] Kitchen Temperature: 93°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [15:15:38] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 15:15:39
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 36°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: FIRE ALERT: Smoke detected in Bedroom!
[LOG] [15:15:39] Sending notification to homeowner...

Time: 15:15:40
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [15:15:40] Sending notification to homeowner...
[READING] Kitchen Temperature: 100°F (Abnormal)
[ALERT!] 🚨 HIGH: SAFETY: T